<a href="https://colab.research.google.com/github/aadhilahamath07-pixel/AlCu-Laser-Welding-ML/blob/main/notebooks/03_build_conventional_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!git clone https://github.com/aadhilahamath07-pixel/AlCu-Laser-Welding-ML.git

%cd /content/AlCu-Laser-Welding-ML

Cloning into 'AlCu-Laser-Welding-ML'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 54 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 38.99 KiB | 4.87 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/AlCu-Laser-Welding-ML


# 03 — Build the First Standardized Dataset

This notebook prepares the conventional laser-welding records for later modelling. It does **not** combine different studies, delete zero measurements, average repeats, or overwrite raw data.

**Input:** `data/raw/Laser Welding.xlsx`  
**Output:** `data/processed/conventional_laser_welding_standardized.csv`


In [9]:
from pathlib import Path
import pandas as pd

RAW_FILE = Path('data/raw/Laser Welding.xlsx')
OUT_DIR = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError(f'Cannot find {RAW_FILE}. Check that the repository files are available in this Colab session.')

raw = pd.read_excel(RAW_FILE, sheet_name='Data')
print('Rows:', len(raw))
print('Original columns:', raw.columns.tolist())
display(raw.head())


Rows: 27
Original columns: ['Laser Power [W]', 'Welding Speed [mm/s]', 'Weld bead width [mm]', 'Interface width [mm]', 'Weld bead penetration depth [mm]']


,Laser Power [W],Welding Speed [mm/s],Weld bead width [mm],Interface width [mm],Weld bead penetration depth [mm]
0,500,200,1.25,0.48,0.27
1,500,200,1.15,0.27,0.30
2,500,200,1.28,0.35,0.10
3,500,450,0.00,0.00,0.00
4,500,450,0.00,0.00,0.00


## 1. Map the source columns

We retain the source measurements and assign consistent names for analysis. If a source heading changes, this cell will stop rather than silently guessing.

In [10]:
column_map = {
    'Laser Power [W]': 'laser_power_W',
    'Welding Speed [mm/s]': 'welding_speed_mm_s',
    'Weld bead width [mm]': 'weld_bead_width_mm',
    'Interface width [mm]': 'interface_width_mm',
    'Weld bead penetration depth [mm]': 'penetration_depth_mm',
}

missing_columns = [c for c in column_map if c not in raw.columns]
if missing_columns:
    raise ValueError(f'Expected source columns not found: {missing_columns}')

df = raw[list(column_map)].rename(columns=column_map).copy()
df.insert(0, 'source_id', 'conventional_laser_welding_dataset')
df.insert(1, 'study_id', 'source_dataset_01')
df.insert(2, 'record_id', [f'CLW_{i+1:03d}' for i in range(len(df))])

# Replicate number within each identical process-parameter combination.
df['replicate_id'] = (
    df.groupby(['laser_power_W', 'welding_speed_mm_s']).cumcount() + 1
)
df['weld_observation_zero_flag'] = (
    df[['weld_bead_width_mm', 'interface_width_mm', 'penetration_depth_mm']]
      .eq(0).all(axis=1)
)
df['data_status'] = 'transcribed_from_source_dataset; zero status not yet verified'

display(df.head(10))


,source_id,study_id,record_id,laser_power_W,welding_speed_mm_s,weld_bead_width_mm,interface_width_mm,penetration_depth_mm,replicate_id,weld_observation_zero_flag,data_status
0,conventional_laser_welding_dataset,source_dataset_01,CLW_001,500,200,1.25,0.48,0.27,1,False,transcribed_from_source_dataset; zero status n...
1,conventional_laser_welding_dataset,source_dataset_01,CLW_002,500,200,1.15,0.27,0.30,2,False,transcribed_from_source_dataset; zero status n...
2,conventional_laser_welding_dataset,source_dataset_01,CLW_003,500,200,1.28,0.35,0.10,3,False,transcribed_from_source_dataset; zero status n...
3,conventional_laser_welding_dataset,source_dataset_01,CLW_004,500,450,0.00,0.00,0.00,1,True,transcribed_from_source_dataset; zero status n...
4,conventional_laser_welding_dataset,source_dataset_01,CLW_005,500,450,0.00,0.00,0.00,2,True,transcribed_from_source_dataset; zero status n...
5,conventional_laser_welding_dataset,source_dataset_01,CLW_006,500,450,0.00,0.00,0.00,3,True,transcribed_from_source_dataset; zero status n...
6,conventional_laser_welding_dataset,source_dataset_01,CLW_007,500,700,0.00,0.00,0.00,1,True,transcribed_from_source_dataset; zero status n...
7,conventional_laser_welding_dataset,source_dataset_01,CLW_008,500,700,0.00,0.00,0.00,2,True,transcribed_from_source_dataset; zero status n...
8,conventional_laser_welding_dataset,source_dataset_01,CLW_009,500,700,0.00,0.00,0.00,3,True,transcribed_from_source_dataset; zero status n...
9,conventional_laser_welding_dataset,source_dataset_01,CLW_010,750,200,0.80,0.22,0.30,1,False,transcribed_from_source_dataset; zero status n...


## 2. Run quality checks

These checks report issues; they do not automatically modify the measurements.

In [11]:
print('Shape:', df.shape)
print('Missing values by column:')
display(df.isna().sum().rename('missing_count').to_frame())

print('Zero-output records:', int(df['weld_observation_zero_flag'].sum()))
print('Records by process condition:')
display(df.groupby(['laser_power_W', 'welding_speed_mm_s']).size().rename('replicate_count').reset_index())

print('Duplicate complete measurement rows:', int(df.duplicated().sum()))
print('No rows have been removed or averaged.')


Shape: (27, 11)
Missing values by column:


,missing_count
source_id,0
study_id,0
record_id,0
laser_power_W,0
welding_speed_mm_s,0
weld_bead_width_mm,0
interface_width_mm,0
penetration_depth_mm,0
replicate_id,0
weld_observation_zero_flag,0


Zero-output records: 6
Records by process condition:


,laser_power_W,welding_speed_mm_s,replicate_count
0,500,200,3
1,500,450,3
2,500,700,3
3,750,200,3
4,750,450,3
5,750,700,3
6,1000,200,3
7,1000,450,3
8,1000,700,3


Duplicate complete measurement rows: 0
No rows have been removed or averaged.


## 3. Save the standardized dataset

The raw Excel file remains untouched. This CSV is a derived working copy with traceability columns.

In [12]:
output_file = OUT_DIR / 'conventional_laser_welding_standardized.csv'
df.to_csv(output_file, index=False)
print('Saved:', output_file)
print('Rows saved:', len(pd.read_csv(output_file)))
print('Raw source unchanged:', RAW_FILE.exists())


Saved: data/processed/conventional_laser_welding_standardized.csv
Rows saved: 27
Raw source unchanged: True


## 4. Download the output from Colab

The file is saved in the current Colab runtime, not automatically in GitHub. To download it, run:

In [14]:
from google.colab import files
files.download(str(output_file))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>